# 2단계: 단어(Word) 단위 토크나이저

## 이 노트북에서 배우는 것

1단계에서 만든 글자 단위 토크나이저는 OOV가 없다는 장점이 있었지만,  
`"hello"` 한 단어를 `['h','e','l','l','o']` 다섯 개의 토큰으로 쪼갠다는 큰 단점이 있습니다.

이 노트북에서는 **단어(Word) 단위** 토크나이저를 직접 만들어보고,  
그 방법이 가진 두 가지 치명적인 한계를 몸소 경험합니다.

| 비교 항목 | 글자 단위 | 단어 단위 |
|-----------|-----------|----------|
| OOV 발생 | 거의 없음 | 자주 발생 |
| vocab 크기 | 작음 (~100개) | 매우 큼 (수십만 개) |
| 의미 표현 | 약함 | 강함 |
| 효율 | 낮음 (토큰 수 많음) | 높음 (토큰 수 적음) |

> 이 두 방법의 장점만 취한 것이 다음 단계에서 배울 **BPE** 입니다.

---
## 1단계: 글자 단위의 문제를 다시 확인

같은 문장을 글자 단위와 단어 단위로 쪼개면 토큰 수가 얼마나 다른지 직접 확인해 봅시다.

In [ ]:
sentence = "I love natural language processing"

# 글자 단위로 쪼개기
char_tokens = list(sentence)
print(f"글자 단위: {len(char_tokens)}개 토큰")
print(f"토큰 목록: {char_tokens}")

print()

# 단어 단위로 쪼개기 (공백 기준)
word_tokens = sentence.split()
print(f"단어 단위: {len(word_tokens)}개 토큰")
print(f"단어 목록: {word_tokens}")

---
## 2단계: 직접 단어 어휘집 만들기

글자 단위와 원리는 동일합니다.  
다른 점은 **글자** 대신 **단어**(공백으로 구분된 단위)를 어휘집에 등록한다는 것입니다.

특수 토큰 `[PAD]`, `[UNK]`, `[BOS]`, `[EOS]` 는 0번부터 먼저 등록합니다.

In [ ]:
# 공백으로 분리 → 어휘집 구축 (클래스 없이)
train_text = "I love machine learning and natural language processing"
words = train_text.split()             # 공백 기준 분리
unique_words = sorted(set(words))      # 중복 제거 후 정렬

# 특수 토큰 먼저 등록 (PAD=0, UNK=1, BOS=2, EOS=3)
special = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
word_to_id = {tok: i for i, tok in enumerate(special)}
id_to_word = {i: tok for i, tok in enumerate(special)}

# 일반 단어 등록 (4번부터)
for word in unique_words:
    idx = len(word_to_id)
    word_to_id[word] = idx
    id_to_word[idx] = word

print("vocab_size:", len(word_to_id))
print("어휘집 일부:", dict(list(word_to_id.items())[:8]))

---
## 3단계: OOV 문제 직접 체험

단어 단위 토크나이저는 학습 데이터에 없는 단어를 만나면 처리하지 못합니다.  
이것이 글자 단위보다 훨씬 심각한 OOV 문제입니다.

신조어(`ChatGPT`), 전문 용어(`transformer`), 오타 등은 어휘집에 없을 가능성이 높습니다.

In [ ]:
# OOV 발생 시나리오: 학습 이후에 생긴 단어를 인코딩 시도
oov_text = "I love ChatGPT and transformer architecture"
words_to_encode = oov_text.split()

ids = []
oov_list = []
for word in words_to_encode:
    if word in word_to_id:
        ids.append(word_to_id[word])
    else:
        ids.append(1)          # UNK_ID = 1
        oov_list.append(word)

print(f"인코딩 결과: {ids}")
print(f"OOV 단어: {oov_list}")
print(f"OOV 비율: {len(oov_list)/len(words_to_encode):.1%}")

# 디코딩하면 UNK가 그대로 남아 원래 단어를 복원할 수 없음
decoded = ' '.join(id_to_word.get(i, '[UNK]') for i in ids)
print(f"디코딩 결과: '{decoded}'")
print("→ 원래 단어 복원 불가!")

---
## 4단계: vocab_size 폭발 직접 확인

단어 단위 토크나이저의 또 다른 문제는 **어휘집 크기가 너무 커진다**는 것입니다.

- 영어 단어: 약 170,000개
- 한국어 활용형 포함: 수백만 개 이상

문장이 하나씩 추가될 때마다 vocab이 어떻게 커지는지 직접 관찰해 봅시다.

In [ ]:
# 문장을 하나씩 추가할 때마다 vocab이 커지는 것을 관찰
sentences = [
    "the cat sat on the mat",
    "the dog ran across the park near the river",
    "a quick brown fox jumped over the lazy dog",
    "she sells sea shells by the sea shore",
    "how much wood would a woodchuck chuck if a woodchuck could chuck wood",
]

vocab = set(["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
print(f"초기 vocab_size (특수토큰만): {len(vocab)}")

for i, sent in enumerate(sentences):
    vocab.update(sent.split())
    print(f"{i+1}번째 문장 추가 후 vocab_size: {len(vocab)}")

print(f"\n영어 단어 전체: ~170,000개")
print(f"한국어 활용형 포함: 수백만 개")

---
## 5단계: 배치 인코딩 + PAD 패딩 직접 구현

언어 모델은 여러 문장을 **배치(Batch)** 단위로 처리합니다.  
그런데 각 문장의 길이가 다르면 행렬로 묶을 수 없습니다.

`[PAD]` 토큰으로 짧은 문장의 빈 자리를 채워서 모든 문장을 **동일한 길이**로 만듭니다.

In [ ]:
# PAD 패딩 직접 구현해보기
texts = ["I love AI", "hello", "natural language"]
batch_ids = []

# 각 문장을 인코딩 (OOV는 UNK=1 으로 대체)
for text in texts:
    ids = []
    for word in text.split():
        ids.append(word_to_id.get(word, 1))  # OOV → UNK
    batch_ids.append(ids)

# 패딩 전: 길이가 제각각
print("패딩 전:")
for i, ids in enumerate(batch_ids):
    print(f"  문장 {i+1}: {ids}  길이 {len(ids)}")

# PAD(=0)로 채워서 가장 긴 문장의 길이에 맞추기
max_len = max(len(ids) for ids in batch_ids)
padded = [ids + [0] * (max_len - len(ids)) for ids in batch_ids]

print("\n패딩 후 (PAD=0):")
for i, ids in enumerate(padded):
    print(f"  문장 {i+1}: {ids}  길이 {len(ids)}")

---
## 6단계: WordTokenizer 클래스 사용

위에서 손으로 한 모든 작업이 `word_tokenizer.py` 에 클래스로 정리되어 있습니다.  
파일을 열어보면 낯선 코드가 아닌, 방금 한 것들의 정리된 버전임을 확인할 수 있습니다.

`WordTokenizer` 에는 OOV 단어를 리포트해주는 특별한 메서드도 있습니다.

In [ ]:
# 클래스 import 및 전체 기능 테스트
import sys, os

# 현재 노트북이 있는 폴더(stage2_word)를 파이썬 경로에 추가
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from word_tokenizer import WordTokenizer

tok = WordTokenizer()
tok.train("I love machine learning and natural language processing")

# OOV 포함 인코딩 + 리포트
report = tok.encode_with_oov_report("I love ChatGPT and transformer")
print(f"인코딩: {report['ids']}")
print(f"OOV 단어: {report['oov_words']}")
print(f"OOV 비율: {report['oov_rate']:.1%}")

In [ ]:
# JSON 저장 및 불러오기
import json

save_path = "word_tokenizer.json"
tok.save(save_path)

# 새 인스턴스로 불러와서 동일한 결과를 내는지 확인
tok2 = WordTokenizer()
tok2.load(save_path)

ids1 = tok.encode("I love language")
ids2 = tok2.encode("I love language")
print(f"원본:       {ids1}")
print(f"불러온 것:  {ids2}")
print(f"동일한가? {ids1 == ids2}")

---
## 정리: 단어 단위의 두 가지 치명적 한계

오늘 직접 경험한 두 가지 문제:

### 1. OOV 문제 — 학습에 없는 단어를 처리 못함
- 신조어, 전문 용어, 오타 등은 어휘집에 없어서 모두 `[UNK]` 으로 대체됩니다
- `[UNK]` 으로 바꾸면 디코딩 시 원래 단어를 **영원히** 복원할 수 없습니다

### 2. vocab_size 폭발 — 어휘집이 너무 커짐
- 영어만 해도 ~170,000개, 한국어는 활용형 포함 수백만 개
- vocab이 크면 모델의 **임베딩 테이블** 크기가 폭발적으로 커집니다
- 메모리와 계산 비용이 감당하기 어려워집니다

---

## 다음 단계 예고: 3단계 — BPE 토크나이저

**BPE(Byte Pair Encoding)** 는 이 두 문제를 동시에 해결합니다:

- 글자(바이트) 단위에서 시작하므로 **OOV가 원천적으로 불가능** (글자 단위의 장점)
- 자주 나오는 글자 쌍을 하나로 합치므로 **토큰 수가 줄어들고 vocab이 제한됨** (단어 단위의 장점)

다음 노트북에서 BPE 알고리즘을 한 줄씩 직접 구현해봅니다.